# 14.5 怎麼讓特徵控制另一組特徵？


一個特徵很強，不代表在每段文字裡都應使用它。例如模型辨認出「可能是人名」的訊號，但上下文顯示這是商品名稱，便需要另一組訊號調整它。普通MLP先算出內容，再經[啟動函數](https://birdhackor.github.io/tiny-perceptron-vlm/14.4.html)改變大小；我們也能準備兩條路，一條產生內容，另一條根據同一個輸入決定每項內容要放大、縮小或反號。後者叫gate（閘門），逐項相乘的組合叫gating。

先用確定的數字把這件事拆開。內容路徑輸出 `[2,2,2]`，閘門路徑輸出 `[-2,0,2]`。我們先對閘門套SiLU：它把x乘上 `sigmoid(x)`，而sigmoid把任意數字變成0到1之間。-2的sigmoid約0.1192，SiLU約-0.2384；0變0；2變1.7616。再與內容逐項相乘，得到約 `[-0.4768,0,3.5232]`。第一項反號、第二項消失、第三項變大。

這個組合稱為SwiGLU，意指使用SiLU的閘門式線性單元。名稱裡的閘門容易讓人誤以為最後係數是0到1的機率；其實SiLU仍含原來的x，輸出可以負、也可以大於1。完整層先把寬度d的輸入投影成寬度h的閘門和內容，兩者相乘後，再投影回d，讓後面的模型維持原形狀。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
import torch.nn.functional as F

content = torch.tensor([2.0, 2.0, 2.0])
gate = torch.tensor([-2.0, 0.0, 2.0])
factor = F.silu(gate)
print("閘門係數", factor.round(decimals=4).tolist())
print("調節內容", (content * factor).round(decimals=4).tolist())
d, h = 8, 16
print("普通MLP權重數", d * h + h * d)
print("SwiGLU權重數", d * h + d * h + h * d)

前三個tensor是可追蹤的小例子；`content * factor`是相同位置逐項乘，不是矩陣乘法。兩行輸出應是上述係數和內容。最後兩行不建立隨機模型，而直接數矩陣裡的數字：普通MLP有 `d×h` 和 `h×d` 兩張權重表，得到256；SwiGLU多了一張 `d×h` 的閘門表，得到384。這裡不使用bias（每項額外加的常數），所以數量只算矩陣權重。

同樣h不等於同樣模型大小。若要公平比較，可把SwiGLU的h減到普通MLP的約三分之二，讓三張表的總數接近原來兩張表，再分別訓練並量測。這也說明架構設計常需要同時考慮「函數做了什麼」與「花多少參數和計算」，不能只把零件換上去就宣稱有收益。

練習把gate改成全零，先預測內容是否仍為 `[2,2,2]`，再核對輸出。接著只改第一項為2，確認只有第一項被放大。你不需要訓練模型，就能先分辨逐特徵控制與把整個向量乘同一音量的差別。
